# Weeks 3+ — Working with the full release (~79M rows) without downloading 79M rows

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/notebooks/03_working_with_the_full_release.ipynb)

Notebooks 01–02 used the small starter CSV that ships with this repo. Your lane and capstone work
run on the **full pseudonymized warehouse release**: ~17 months of daily search performance for
~70 clients, plus a query-level table. It is hosted as Parquet on Hugging Face, and the trick of
this notebook is that you **never download or load the whole thing** — DuckDB reads only the
columns and partitions your SQL touches.

By the end you will have:
1. Connected DuckDB to the hosted release and listed every table.
2. Pulled a **feature table you designed** (aggregates per content item) into pandas.
3. Trained a quick scikit-learn model on features you built from 79M rows — on a free Colab CPU.

**Before you start (one-time, ~2 minutes):**
1. Create a free [Hugging Face account](https://huggingface.co/join).
2. Open the dataset page ([`FlyRank/internship-warehouse`](https://huggingface.co/datasets/FlyRank/internship-warehouse)) and **request access** (instant after you accept the data-use terms). **Accept the terms in your browser first — the token below 401s until access is granted (usually instant).**
3. Create a **read** token at [Settings → Access Tokens](https://huggingface.co/settings/tokens). **Never paste the token into a code cell** — your repo is public; use the `getpass` prompt below (or Colab's 🔑 Secrets panel).


In [1]:
%pip -q install duckdb huggingface_hub


Note: you may need to restart the kernel to use updated packages.


In [3]:
import os, getpass

# CI and power users set HF_TOKEN in the environment; everyone else gets the safe prompt.
HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')


## 1. Connect DuckDB to the release

DuckDB speaks `hf://` natively. The secret below authenticates every query; after that the
release behaves like a set of local tables.


In [4]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')


dim_clients                     104 rows
dim_content                 519,606 rows
fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


That count over the daily fact touched **Parquet metadata, not data** — it finished in seconds
even though the table has ~79M rows. That is the whole workflow: push the heavy lifting into
DuckDB SQL, bring only small results into pandas.

## 2. Know your panel before you model it

History depth **differs per client** (an *unbalanced panel*). `dim_clients` tells you exactly
what each client has — check it before designing any time window.


In [5]:
clients = con.sql(f"""
    SELECT client_hash_id, access_profile, gsc_data_start, ga4_data_start
    FROM {TABLES['dim_clients']}
    ORDER BY gsc_data_start NULLS LAST
""").df()

print('clients with 12+ months of GSC history:',
      (clients['gsc_data_start'] <= clients['gsc_data_start'].dropna().max() - __import__('pandas').Timedelta(days=365)).sum())
clients.head(10)


clients with 12+ months of GSC history: 4


,client_hash_id,access_profile,gsc_data_start,ga4_data_start
0,client_9958f0a7ae1df715,gsc_and_ga4,2025-01-27,2025-10-29
1,client_ff644d8251367cbb,gsc_and_ga4,2025-01-27,2025-10-29
2,client_73cda7b4e4f265ea,gsc_and_ga4,2025-02-11,2026-03-24
3,client_fef1a8f436438636,gsc_and_ga4,2025-03-11,2026-03-06
4,client_62f4a7e64f5e0096,gsc_only,2025-06-07,NaT
5,client_b10cb2997d0c7c86,gsc_and_ga4,2025-06-18,2025-11-15
6,client_c182d11e4862a37d,gsc_and_ga4,2025-06-21,2026-02-20
7,client_65de48885f4ef01b,gsc_and_ga4,2025-06-21,2026-02-19
8,client_3197e6291363b4db,gsc_and_ga4,2025-06-29,2025-11-09
9,client_625b6439094e23e4,gsc_and_ga4,2025-07-01,2026-02-19


## 3. Build features with SQL, not with RAM

The pattern for every lane: **aggregate per content item inside DuckDB**, then hand the small
result to pandas/sklearn. Here: momentum features from the last 60 days of the panel.

**This is the heaviest cell in the notebook — expect 2–6 minutes on Colab.** It downloads ~2 months of column data over the network (RAM stays tiny; that's the point). If it runs past ~10 minutes or errors with `HTTP 429`, re-run this section against `TABLES['fact_daily_sample']` and save the full table for your final pass.


In [6]:
features = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}
    ),
    windowed AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_last30,
               AVG(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_avg_position END)       AS pos_last30
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL 60 DAY
        GROUP BY 1, 2
        HAVING imp_prev30 >= 100
    )
    SELECT * FROM windowed
""").df()

print(f'{len(features):,} content items with enough history')
features.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

111,247 content items with enough history


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30
0,client_e547b89c05043229,content_ec8dd59de31c34b4,199.0,443.0,0.0,16.462961
1,client_e547b89c05043229,content_fe05ee94ac2887c4,194.0,304.0,0.0,47.810624
2,client_e547b89c05043229,content_f61c62e31bd4af4c,291.0,827.0,1.0,6.284019
3,client_e547b89c05043229,content_b7fd0503ddf6bcc4,163.0,202.0,0.0,10.713791
4,client_e547b89c05043229,content_f28da95f06529bb4,1120.0,788.0,7.0,13.067961


## 4. Add query-level signals

`fact_content_query_90d` describes **how a page earns its impressions**: across how many
distinct queries, how concentrated, how much sits in the rare/anonymized tail. One page ranking
for 40 queries is a different animal from one page ranking for 2.


In [7]:
qsignals = con.sql(f"""
    SELECT content_hash_id,
           ANY_VALUE(content_visible_query_count)     AS visible_queries,
           ANY_VALUE(rare_impressions_share)          AS rare_share,
           ANY_VALUE(anonymized_impressions_share)    AS anon_share,
           MAX(impressions_90d)                       AS top_query_impressions,
           SUM(impressions_90d)                       AS kept_impressions
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()

qsignals['top_query_share'] = qsignals['top_query_impressions'] / qsignals['kept_impressions']
data = features.merge(qsignals, on='content_hash_id', how='left')
print(f'joined: {len(data):,} rows')
data.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

joined: 111,247 rows


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30,visible_queries,rare_share,anon_share,top_query_impressions,kept_impressions,top_query_share
0,client_e547b89c05043229,content_ec8dd59de31c34b4,199.0,443.0,0.0,16.462961,12.0,0.090708,0.573009,214.0,456.0,0.469298
1,client_e547b89c05043229,content_fe05ee94ac2887c4,194.0,304.0,0.0,47.810624,9.0,0.381953,0.422744,31.0,158.0,0.196203
2,client_e547b89c05043229,content_f61c62e31bd4af4c,291.0,827.0,1.0,6.284019,17.0,0.095365,0.779593,58.0,375.0,0.154667
3,client_e547b89c05043229,content_b7fd0503ddf6bcc4,163.0,202.0,0.0,10.713791,4.0,0.164887,0.752076,26.0,70.0,0.371429
4,client_e547b89c05043229,content_f28da95f06529bb4,1120.0,788.0,7.0,13.067961,14.0,0.101023,0.812020,69.0,272.0,0.253676


## 5. A first honest model

Same shape as notebook 02: define a label, hold out data, compare against a dumb baseline.
Label: *did impressions decline by more than 20% month-over-month?* — built only from columns
that exist **before** the window we predict. (Momentum features from the last 30 days predicting
a label defined on those same 30 days would be leakage — so here the features come from the
prev-30 window and query-mix, and the label from the last-30 outcome.)


In [8]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

data['is_declining'] = (data['imp_last30'] < 0.8 * data['imp_prev30']).astype(int)

feature_cols = ['imp_prev30', 'visible_queries', 'rare_share', 'anon_share', 'top_query_share']
model_data = data.dropna(subset=feature_cols)
X, y = model_data[feature_cols], model_data['is_declining']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr, y_tr)

print(f'base rate (always predict majority): {max(y_te.mean(), 1 - y_te.mean()):.3f}')
print(classification_report(y_te, model.predict(X_te), digits=3))

base rate (always predict majority): 0.633
              precision    recall  f1-score   support

           0      0.543     0.332     0.412      9389
           1      0.683     0.837     0.753     16162

    accuracy                          0.652     25551
   macro avg      0.613     0.585     0.582     25551
weighted avg      0.632     0.652     0.628     25551



Whatever number you just got: interrogate it before you believe it. Which feature carries the
signal? Does it survive a per-client split (train on some clients, test on others)? That
question — *does it generalize across clients?* — is exactly what separates a capstone-grade
result from a lucky split.

## Your turn

1. Re-run section 3 with a **90-day** window and a `HAVING` threshold of your choice.
2. Add one feature you believe in (position volatility? weekend share? query concentration?).
3. Replace the random split with **GroupShuffleSplit on `client_hash_id`** and compare.

## Working locally instead

```python
from huggingface_hub import snapshot_download
path = snapshot_download(repo_id='FlyRank/internship-warehouse', repo_type='dataset',
                         allow_patterns=['dim_*.parquet', 'fact_content_query_90d.parquet',
                                         'fact_content_daily_performance/month=2026-0*/*.parquet'])
```
Then point `REL` at that local path. Download only the month partitions you need — the
`allow_patterns` filter above is the whole trick.

---

**Where this fits:** every lane brief assumes you can produce per-content feature tables like
the one you just built. The lane datasets under the `lanes` HF repo are pre-cut examples of
exactly this pattern — but for the capstone, features you engineered yourself from the full
release beat any pre-cut file.


In [13]:
features = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}
    ),
    windowed AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_last30,
               AVG(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_avg_position END)       AS pos_last30
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL 60 DAY
        GROUP BY 1, 2
        HAVING imp_prev30 >= 100
    )
    SELECT * FROM windowed
""").df()

print(f'{len(features):,} content items with enough history')
features.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

111,247 content items with enough history


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30
0,client_20259bd6705d81d4,content_ea371636ef45d77f,36.0,113.0,0.0,26.162281
1,client_20259bd6705d81d4,content_8c8ba2dd1e855cc8,6.0,101.0,0.0,17.200000
2,client_20259bd6705d81d4,content_fb7806f3783f1400,38.0,847.0,0.0,5.634921
3,client_20259bd6705d81d4,content_36d529df2af1b6a2,21.0,106.0,0.0,9.820513
4,client_20259bd6705d81d4,content_114936f1929c7631,43.0,119.0,0.0,5.751515


In [17]:
features_with_volatility = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}
    ),
    windowed AS (
        SELECT f.client_hash_id, 
               f.content_hash_id,
               -- Base metrics (Prior 91 to 180 days ago)
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 90 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev90,
               AVG(CASE WHEN f.report_date <= b.end_d - INTERVAL 90 DAY THEN f.gsc_avg_position END) AS pos_prev90,
               
               -- Engineered Feature: Position Volatility (Standard Deviation during feature window)
               COALESCE(STDDEV_SAMP(CASE WHEN f.report_date <= b.end_d - INTERVAL 90 DAY THEN f.gsc_avg_position END), 0) AS pos_volatility_prev90,
               
               -- Target metrics (Most recent 90 days)
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 90 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last90,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 90 DAY THEN f.gsc_clicks ELSE 0 END) AS clk_last90,
               AVG(CASE WHEN f.report_date >  b.end_d - INTERVAL 90 DAY THEN f.gsc_avg_position END) AS pos_last90
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL 180 DAY
        GROUP BY 1, 2
        HAVING imp_prev90 >= 500
    )
    SELECT *,
           -- Simple proxy target: Active decline defined as a >10% drop in impressions
           CAST((imp_last90 - imp_prev90) / NULLIF(imp_prev90, 0) <= -0.10 AS INTEGER) AS is_declining_label
    FROM windowed
""").df()
features_with_volatility.head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,imp_prev90,pos_prev90,pos_volatility_prev90,imp_last90,clk_last90,pos_last90,is_declining_label
0,client_c182d11e4862a37d,content_c814de8d96a68954,508.0,9.197280,3.642913,0.0,0.0,NaN,1
1,client_c182d11e4862a37d,content_ca47029b2fb29d12,7917.0,6.147492,2.564990,3895.0,11.0,5.724692,1
2,client_c182d11e4862a37d,content_34b8de016805d44d,740.0,12.335418,5.345295,0.0,0.0,NaN,1
3,client_400c21c81c8b46ef,content_e79e3b514053fc92,813.0,8.395675,4.982209,2298.0,1.0,14.121068,0
4,client_400c21c81c8b46ef,content_6fd07877d065131f,583.0,7.402402,5.013514,68.0,0.0,7.328654,1
5,client_400c21c81c8b46ef,content_cc113e169c79be51,879.0,6.925867,3.586700,123.0,0.0,5.336380,1
6,client_400c21c81c8b46ef,content_763c874513e43b25,1524.0,7.162062,2.461588,3238.0,2.0,36.647877,0
7,client_400c21c81c8b46ef,content_f9bac254a25e72af,805.0,11.801705,7.335176,0.0,0.0,NaN,1
8,client_400c21c81c8b46ef,content_4ff3b27c75eed57b,962.0,7.713850,3.400715,706.0,2.0,12.889163,1
9,client_400c21c81c8b46ef,content_1f0a360ee5ea2261,551.0,8.768481,7.868381,15.0,0.0,5.369048,1


In [18]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

# 1. Define Features (X), Labels (y), and Groups (groups)
X = features_with_volatility[["imp_prev90", "pos_prev90", "pos_volatility_prev90"]]
y = features_with_volatility["is_declining_label"]
groups = features_with_volatility["client_hash_id"]

# 2. Implement GroupShuffleSplit (Hold out 20% of clients completely)
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

# Print proof of isolation
train_clients = set(groups.iloc[train_idx])
test_clients = set(groups.iloc[test_idx])
print(f"Clients in Train: {len(train_clients)} | Clients in Test: {len(test_clients)}")
print(f"Any overlap? {not train_clients.isdisjoint(test_clients)}")

# 3. Fit and Evaluate Model
clf = RandomForestClassifier(random_state=42)
clf.fit(X_train, y_train)

preds = clf.predict(X_test)
probs = clf.predict_proba(X_test)[:, 1]

print("\n--- Model Evaluation (Group Holdout Split) ---")
print(f"ROC AUC Score: {roc_auc_score(y_test, probs):.3f}")
print(classification_report(y_test, preds))

Clients in Train: 32 | Clients in Test: 8
Any overlap? False

--- Model Evaluation (Group Holdout Split) ---
ROC AUC Score: 0.669
              precision    recall  f1-score   support

           0       0.42      0.56      0.48      6111
           1       0.79      0.69      0.73     15054

    accuracy                           0.65     21165
   macro avg       0.61      0.62      0.61     21165
weighted avg       0.68      0.65      0.66     21165

